In [2]:
import numpy as np

# Primary task

In [3]:
def arrivals_in_day(rate, t, Idx_for_process): 
    # Input: rate for arrival time, t is the day in the year, Idx_for_process is the type of patient
    # Output: list of a tuples with patient type in first entry and arrivaltime in the second entry. 


    # Initialize start of day and patients.
    time = 0
    patients = []

    # Let 0 patients arrive if rate is 0
    if rate <=0: 
        return []

    
    while True:
        time += np.random.exponential(1 / rate)

        # Check we are still within one day
        if time > 1:
            break

        # Append patient type and time for arrival
        patients.append((Idx_for_process, t + time))


    return patients

def arrivals_year(lam1, lam2,lam3):
    # Input: lami is the arrival rate function for ward i. 
    # Output: A list of tuples where the first entry in the tuple is the patient type and the last entry is the arrival time.
    
    # Initialize
    t =0
    Patients_1 = []
    Patients_2 = []
    Patients_3 = []

    #Iterate over the days
    while t < 365: 
        # Find rates
        rate1 = lam1(t)
        rate2 = lam2(t)
        rate3 = lam3(t)

        # Simulate arrivals for all three patient types. 
        Patients_1.extend(arrivals_in_day(rate1,t,1))
        Patients_2.extend(arrivals_in_day(rate2,t,2))
        Patients_3.extend(arrivals_in_day(rate3,t,3))

        t+=1

    # Merge list to create one list of all arrivals in a year
    All_patients = sorted(Patients_1 + Patients_2 + Patients_3, key=lambda x: x[1])
    return All_patients


In [4]:
def lam1(t): 
   return -(1/3650)*t**2 + (1/10)*t

def lam2(t): 
   return lam1(t)/5

def lam3(t):
   return 6


X = arrivals_year(lam1,lam2,lam3)

In [5]:
X[-1]

(3, 364.80628751093406)

In [6]:
# System of wards
def system(bedsA,bedsB,bedsC, patientflow_year):
    # Input: bedsA is number of beds in ward A,bedsB is number of beds in ward B, bedsC is number of beds in ward C. Patient_flow_year is a list of patients arriving in a year, where each entry in the list is a tuple containing the patient type and their arrival time. 
    # Output: blocked_i is the number of relocated patients for ward i and np.mean(bed_frac_i)/bedsi is the mean value of the fraction of beds in use in ward i

    #Initialize
    blocked_A =0
    blocked_B = 0
    blocked_C = 0

    beds_A = np.zeros(bedsA)
    beds_B = np.zeros(bedsB)
    beds_C = np.zeros(bedsC)

    bed_frac_A = []
    bed_frac_B = []
    bed_frac_C = []

    # Iterate through all patients
    for type, t in patientflow_year:
        # Release beds if time has passed of arrivaltime+LOS
        beds_A[beds_A <= t] = 0
        beds_B[beds_B <= t] = 0
        beds_C[beds_C <= t] = 0


        # Find idle beds
        idle_beds_A = np.where(beds_A == 0)[0]
        idle_beds_B = np.where(beds_B == 0)[0]
        idle_beds_C = np.where(beds_C == 0)[0]

        # Append number of beds in use
        bed_frac_A.append(bedsA-len(idle_beds_A))
        bed_frac_B.append(bedsB-len(idle_beds_B))
        bed_frac_C.append(bedsC-len(idle_beds_C))


        # Patients in ward A
        if type ==1: 
        # Find Length-of-Stay
            LOS = np.random.lognormal(np.log(4*np.sqrt(2)),np.log(2))
            # Check for idle beds
            if len(idle_beds_A) > 0:
                bed_id = idle_beds_A[0]
                beds_A[bed_id] = t + LOS

            else:
                # Reallocate patient
                blocked_A += 1
            
        # Patients in ward B
        elif type ==2: 
            # Find Length-of-Stay
            LOS = np.random.lognormal(np.log(6*np.sqrt(2)),np.log(2))
            # Check for idle beds
            if len(idle_beds_B) > 0:
                bed_id = idle_beds_B[0]
                beds_B[bed_id] = t + LOS

            else:
                # Increase blocked patients in B, and reallocate patient to A.
                blocked_B += 1
                if len(idle_beds_A)>0:
                    bed_id = idle_beds_A[0]
                    beds_A[bed_id] = t + LOS
                else: 
                    # If no space in A, randomly choose a patient in A to reallocate
                    blocked_A +=1
                    bed_id = np.random.choice(len(beds_A))
                    beds_A[bed_id] = t + LOS


        # Patients in ward C
        else: 
            # Find Length-of-Stay
            LOS = np.random.lognormal(np.log(5*np.sqrt(2)),np.log(2))
            # Check for idle beds
            if len(idle_beds_C) > 0:
                bed_id = idle_beds_C[0]
                beds_C[bed_id] = t + LOS

            else:
                # Reallocate patient
                blocked_C += 1
                
    return blocked_A,blocked_B,blocked_C, np.mean(bed_frac_A)/bedsA,np.mean(bed_frac_B)/bedsB,np.mean(bed_frac_C)/bedsC


# Sensitivity Analysis

In [7]:
# We make a monte carlo estimator just for sum of reallocated patients. 
# This is monte carlo
def sum_relocated(bedsA,bedsB,bedsC,patient_flows):
    # Input: bedsA is number of beds in ward A,bedsB is number of beds in ward B, bedsC is number of beds in ward C.patient_flows is a list of simulated yearly patient flow.
    # Output: mean and variance of the sum of reallocated patients across wards A, B and C
    A = []
    B = []
    C = []

    for X in patient_flows:
        a, b, c, bed_frac_A,bed_frac_B,bed_frac_C = system(bedsA,bedsB,bedsC, X)
        A.append(a)
        B.append(b)
        C.append(c)
    all = np.array(A)+np.array(B)+np.array(C)
    return np.mean(all), np.var(all)

In [8]:
# Modify existing system, in order to get output needed for control variate
def system_control(bedsA,bedsB,bedsC, patientflow_year):
    # Input: bedsA is number of beds in ward A,bedsB is number of beds in ward B, bedsC is number of beds in ward C. Patient_flow_year is a list of patients arriving in a year, where each entry in the list is a tuple containing the patient type and their arrival time. 
    # Output: blocked_i is the number of relocated patients for ward i and LOS_i is the mean value of LOS in use in ward i
    blocked_A =0
    blocked_B = 0
    blocked_C = 0

    beds_A = np.zeros(bedsA)
    beds_B = np.zeros(bedsB)
    beds_C = np.zeros(bedsC)

    LOS_A = []
    LOS_B = []
    LOS_C = []

    for type, t in patientflow_year:
        # Release beds if time has passed
        beds_A[beds_A <= t] = 0
        beds_B[beds_B <= t] = 0
        beds_C[beds_C <= t] = 0

        idle_beds_A = np.where(beds_A == 0)[0]
        idle_beds_B = np.where(beds_B == 0)[0]
        idle_beds_C = np.where(beds_C == 0)[0]

        if type ==1: 
            LOS = np.random.lognormal(np.log(4*np.sqrt(2)),np.log(2))
            if len(idle_beds_A) > 0:
                bed_id = idle_beds_A[0]
                beds_A[bed_id] = t + LOS
                LOS_A.append(LOS)

            else:
                blocked_A += 1
            

        elif type ==2: 
            LOS = np.random.lognormal(np.log(6*np.sqrt(2)),np.log(2))
    
            if len(idle_beds_B) > 0:
                bed_id = idle_beds_B[0]
                beds_B[bed_id] = t + LOS
                LOS_B.append(LOS)

            else:
                blocked_B += 1
                LOS_B.append(LOS)
                # SKal rykke B over i A og hvis A er fuld incremente at nogle er blevet afvist fra A.
                if len(idle_beds_A)>0:
                    bed_id = idle_beds_A[0]
                    beds_A[bed_id] = t + LOS
                    
                else: 
                    # Vælger randomly (er det rigtigt???), hvem der skal smides ud af A...
                    blocked_A+=1
                    bed_id = np.random.choice(len(beds_A))
                    beds_A[bed_id] = t + LOS
        
        else: 
            LOS = np.random.lognormal(np.log(5*np.sqrt(2)),np.log(2))
            if len(idle_beds_C) > 0:
                bed_id = idle_beds_C[0]
                beds_C[bed_id] = t + LOS
                LOS_C.append(LOS)


            else:
                blocked_C += 1
                
    return blocked_A,blocked_B,blocked_C, np.mean(LOS_A),np.mean(LOS_B), np.mean(LOS_C)

# Control variate function
def control_variate_sum(bedsA, bedsB, bedsC,patient_flows):
    # Input: bedsA is number of beds in ward A,bedsB is number of beds in ward B, bedsC is number of beds in ward C. patient_flows is a list of simulated yearly patient flow.
    # Output: mean and variance of the sum of reallocated patients across ward A, B and C. 
    #### Find ci
    # Initialize
    A = []
    B = []
    C = []

    LA = []
    LB = []
    LC = []

    # Iterate
    for X in patient_flows[:50]:
        a,b,c,la,lb,lc = system_control(bedsA,bedsB,bedsC, X)
        A.append(a)
        B.append(b)
        C.append(c)
        LA.append(la)
        LB.append(lb)
        LC.append(lc)
    
    A = np.array(A)
    B = np.array(B)
    C = np.array(C)
    LA = np.array(LA)
    LB = np.array(LB)
    LC = np.array(LC)

    # Find ci
    ca = -np.cov(A,LA)[0,1]/np.var(LA)
    cB = -np.cov(B,LB)[0,1]/np.var(LB)
    cC = -np.cov(C,LC)[0,1]/np.var(LC)

    # Reinitialize to actually find the control variates
    A = []
    B = []
    C = []

    LA = []
    LB = []
    LC = []

    # Iterate
    for X in patient_flows[50:]:
        a,b,c,la,lb,lc = system_control(bedsA,bedsB,bedsC, X)
        A.append(a)
        B.append(b)
        C.append(c)
        LA.append(la)
        LB.append(lb)
        LC.append(lc)
    
    A = np.array(A)
    B = np.array(B)
    C = np.array(C)
    LA = np.array(LA)
    LB = np.array(LB)
    LC = np.array(LC)

    # Find new variable
    Ya = A+ca*(LA-8) #REMEMBER TO CHANGE MEANS IF THEY CHANGE!!!!
    meanA = np.mean(Ya)

    YB = B+cB*(LB-12) #REMEMBER TO CHANGE MEANS IF THEY CHANGE!!!!
    meanB = np.mean(YB)

    YC = C+cC*(LC-10) #REMEMBER TO CHANGE MEANS IF THEY CHANGE!!!!
    meanC = np.mean(YC)



    return meanA+meanB+meanC, np.var(Ya)+np.var(YB)+np.var(YC)+2*np.cov(Ya,YB)[0,1]+2*np.cov(YB,YC)[0,1]+2*np.cov(Ya,YC)[0,1]

In [9]:
np.random.seed(42)
patient_flow = [arrivals_year(lam1, lam2,lam3) for _ in range(100)]
sum_relocated(15,15,45,patient_flow)

(2235.71, 6812.305899999999)

In [11]:

control_variate_sum(15,15,45,patient_flow)

(2453.7928194645788, 4290.07089615875)

In [12]:
# Reduction
(6812- 4290)/6812*100

37.02290076335878